# Transform ML Features From `results-filtered.xlsx`

This notebook reads the `all` sheet from `results-filtered.xlsx` and converts the raw graph columns

- `dataset`
- `k`
- `n`
- `m`
- `d_avg`
- `d_max`
- `core_number`
- `shell_sizes`

into ML-style derived features similar to the earlier feature-engineering code.


In [1]:
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

NOTEBOOK_DIR = Path.cwd()
FILE_PATH = NOTEBOOK_DIR / "results-filtered.xlsx"
SHEET_NAME = "all"

FILE_PATH

PosixPath('/Users/akahmad/Documents/kplex-ML/notebooks/results-filtered.xlsx')

In [2]:
base_cols = [
    "dataset",
    "k",
    "n",
    "m",
    "d_avg",
    "d_max",
    "core_number",
    "shell_sizes",
    "UMKP-P-bnb",
    "UMKP-CTCP-P-bnb",
]

df_raw = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
df_base = df_raw[base_cols].copy()

print(df_base.shape)
df_base.head()

(4356, 8)


,dataset,k,n,m,d_avg,d_max,core_number,shell_sizes
0,12month1,20,52328,1585220,30.294,9214,57,"[(19,52328),(20,48272),(21,44653),(22,41313),(..."
1,130bit,8,584,6059,10.375,322,12,"[(8,584),(9,583),(10,580),(11,566),(12,509)]"
2,130bit,9,584,6059,10.375,322,12,"[(8,584),(9,583),(10,580),(11,566),(12,509)]"
3,130bit,10,583,6051,10.379,321,12,"[(9,583),(10,580),(11,566),(12,509)]"
4,144,4,144319,1072349,7.430,26,9,"[(7,144319),(8,143537),(9,132943)]"


In [3]:
def parse_shell_sizes(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    return literal_eval(text)


def shell_features(shell_value):
    shells = parse_shell_sizes(shell_value)
    if not shells:
        return {
            "core_min": np.nan,
            "shell_num": 0,
            "shell_avgSize": np.nan,
            "shell_maxSize": np.nan,
            "shell_minSize": np.nan,
            "shell_stdSize": np.nan,
        }

    sizes = [size for _, size in shells]
    return {
        "core_min": shells[0][0],
        "shell_num": len(shells),
        "shell_avgSize": float(np.mean(sizes)),
        "shell_maxSize": int(np.max(sizes)),
        "shell_minSize": int(np.min(sizes)),
        "shell_stdSize": float(np.std(sizes)),
    }


In [4]:
feature_rows = df_base["shell_sizes"].apply(shell_features)
shell_df = pd.DataFrame(feature_rows.tolist())

df_features = pd.concat([
    df_base[["dataset", "k", "n", "m", "d_avg", "d_max", "UMKP-P-bnb", "UMKP-CTCP-P-bnb"]].copy(),
    df_base[["core_number"]].rename(columns={"core_number": "core_max"}),
    shell_df,
], axis=1)

df_features.head()

,dataset,k,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize
0,12month1,20,52328,1585220,30.294,9214,57,19,39,16785.102564,52328,1030,14354.722410
1,130bit,8,584,6059,10.375,322,12,8,5,564.400000,584,509,28.443628
2,130bit,9,584,6059,10.375,322,12,8,5,564.400000,584,509,28.443628
3,130bit,10,583,6051,10.379,321,12,9,4,559.500000,583,509,29.853810
4,144,4,144319,1072349,7.430,26,9,7,3,140266.333333,144319,132943,5188.210310


In [5]:
ordered_cols = [
    "dataset",
    "k",
    "n",
    "m",
    "d_avg",
    "d_max",
    "UMKP-P-bnb",
    "UMKP-CTCP-P-bnb",
    "core_max",
    "core_min",
    "shell_num",
    "shell_avgSize",
    "shell_maxSize",
    "shell_minSize",
    "shell_stdSize",
]

df_features = df_features[ordered_cols]
df_features.describe(include="all").transpose()

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
dataset,4356,857,TSC_OPF_1047,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
k,4356.0,NaN,NaN,NaN,8.93618,5.288426,2.0,5.0,8.0,10.0,20.0
n,4356.0,NaN,NaN,NaN,61713.312213,217799.234331,13.0,847.25,8804.0,60271.0,3683678.0
m,4356.0,NaN,NaN,NaN,1428478.598485,7292490.711434,59.0,35473.25,381635.0,1270590.25,135815154.0
d_avg,4356.0,NaN,NaN,NaN,55.708105,106.119128,1.392,18.128,31.659,55.837,1658.049
d_max,4356.0,NaN,NaN,NaN,6302.524105,51038.171678,3.0,80.0,193.5,722.0,1097127.0
core_max,4356.0,NaN,NaN,NaN,99.376951,226.57896,2.0,26.0,44.0,86.0,3280.0
core_min,4356.0,NaN,NaN,NaN,66.315886,204.135583,2.0,14.0,24.0,47.0,3280.0
shell_num,4356.0,NaN,NaN,NaN,22.704545,47.466159,1.0,3.0,7.0,19.0,536.0
shell_avgSize,4356.0,NaN,NaN,NaN,43368.846614,123298.982201,13.0,711.322115,6144.9,51429.5,3683678.0


In [6]:
OUT_CSV = NOTEBOOK_DIR / "results-filtered-ml-features.csv"
OUT_XLSX = NOTEBOOK_DIR / "results-filtered-ml-features.xlsx"

df_features.to_csv(OUT_CSV, index=False)
df_features.to_excel(OUT_XLSX, index=False)

print("Wrote:")
print(OUT_CSV)
print(OUT_XLSX)

Wrote:
/Users/akahmad/Documents/kplex-ML/notebooks/results-filtered-ml-features.csv
/Users/akahmad/Documents/kplex-ML/notebooks/results-filtered-ml-features.xlsx
